# Extract renders_v3 tar.gz files on Google Drive
Extracts all `*.tar.gz` from `HDR_Lab/renders_v3/` into the same directory,
then removes the tar files to free space.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, subprocess, time

tar_dir = '/content/drive/MyDrive/HDR_Lab/renders_v3'
tars = sorted(glob.glob(os.path.join(tar_dir, '*.tar.gz')))
print(f'Found {len(tars)} tar.gz files')

for i, tar_path in enumerate(tars):
    name = os.path.basename(tar_path)
    obj_name = name.replace('.tar.gz', '')
    out_dir = os.path.join(tar_dir, obj_name)
    # Skip if already extracted (12 sessions, each with 200 samples)
    if os.path.isdir(out_dir):
        sessions = [s for s in os.listdir(out_dir) if s.startswith('session_')]
        if len(sessions) == 12:
            samples_dir = os.path.join(out_dir, sessions[0], 'sensor_0000', 'samples')
            if os.path.isdir(samples_dir) and len(os.listdir(samples_dir)) >= 200:
                print(f'[{i+1}/{len(tars)}] {obj_name} already extracted, skipping')
                continue
    print(f'[{i+1}/{len(tars)}] Extracting {name}... ', end='', flush=True)
    t0 = time.time()
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run(
        ['tar', 'xzf', tar_path, '--strip-components=1', '-C', out_dir],
        check=True
    )
    elapsed = time.time() - t0
    print(f'done ({elapsed:.1f}s)')

print(f'\nAll done!')

In [ ]:
# Verify: check sample counts per object
for obj in sorted(os.listdir(tar_dir)):
    obj_path = os.path.join(tar_dir, obj)
    if not os.path.isdir(obj_path) or obj.endswith('.tar.gz'):
        continue
    sessions = [s for s in os.listdir(obj_path) if s.startswith('session_')]
    complete = 0
    for s in sessions:
        samples = os.path.join(obj_path, s, 'sensor_0000', 'samples')
        if os.path.isdir(samples):
            n = len([f for f in os.listdir(samples) if f.endswith('.png')])
            if n == 200:
                complete += 1
    print(f'{obj}: {complete}/{len(sessions)} sessions complete')

In [ ]:
# Optional: remove tar.gz files to free ~10GB
# Uncomment below to delete
# for tar_path in tars:
#     os.remove(tar_path)
#     print(f'Removed {os.path.basename(tar_path)}')
# print('Done! Freed ~10GB')